# 2. Ingredient Detection Module

This module handles ingredient detection using:
- **CLIP** (for ingredient classification)
- **DETR** (for multi-object detection)

**Usage:**
```python
%run 2_ingredient_detection.ipynb
detected = detect_multiple_ingredients_clip(image_path)
```

In [1]:
import torch
import pandas as pd
from pathlib import Path
from typing import Dict, List
from PIL import Image
from transformers import CLIPProcessor, CLIPModel, DetrImageProcessor, DetrForObjectDetection
import warnings
warnings.filterwarnings('ignore')

print("✓ Imports loaded")

✓ Imports loaded


In [2]:
# Paths
PROJECT_ROOT = Path.cwd().parent.parent.parent
DATA_DIR = PROJECT_ROOT / "data"
INGREDIENTS_CSV = DATA_DIR / "ingredients_vocabulary.csv"

# Detection settings
DETECTION_MODE = "multi"  # "single" or "multi"
INGREDIENT_CONFIDENCE_THRESHOLD = 0.15
OBJECT_DETECTION_THRESHOLD = 0.7

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Detection mode: {DETECTION_MODE}")

✓ Data directory: c:\Users\Champion\Documents\GitHub\cAIuldron\data
✓ Detection mode: multi


In [3]:
# Load ingredient vocabulary
if not INGREDIENTS_CSV.exists():
    raise FileNotFoundError(f"Ingredients CSV not found: {INGREDIENTS_CSV}")

df = pd.read_csv(INGREDIENTS_CSV)
INGREDIENT_CANDIDATES = df['Ingredient'].tolist()

print(f"✓ Loaded {len(INGREDIENT_CANDIDATES)} ingredients")

✓ Loaded 528 ingredients


In [4]:
# Load CLIP model
try:
    clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
    clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    print("✓ CLIP model loaded")
except Exception as e:
    print(f"✗ Failed to load CLIP model: {e}")
    clip_model = None
    clip_processor = None

✓ CLIP model loaded


In [5]:
# Load DETR for multi-ingredient detection
if DETECTION_MODE == "multi":
    try:
        detr_processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
        detr_model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")
        print("✓ DETR model loaded (multi-ingredient detection)")
    except Exception as e:
        print(f"✗ Failed to load DETR: {e}")
        detr_model = None
        detr_processor = None
else:
    detr_model = None
    detr_processor = None
    print("⚠️  DETR not loaded (single-ingredient mode)")

Some weights of the model checkpoint at facebook/detr-resnet-50 were not used when initializing DetrForObjectDetection: ['model.backbone.conv_encoder.model.layer1.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked', 'model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked']
- This IS expected if you are initializing DetrForObjectDetection from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DetrForObjectDetection from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


✓ DETR model loaded (multi-ingredient detection)


In [6]:
def detect_ingredient_clip(image_path: str, confidence_threshold: float = 0.15) -> Dict:
    """Detect ingredient using CLIP (single-ingredient mode)"""
    image = Image.open(image_path).convert('RGB')
    
    inputs = clip_processor(
        text=INGREDIENT_CANDIDATES,
        images=image,
        return_tensors="pt",
        padding=True
    )
    
    with torch.no_grad():
        outputs = clip_model(**inputs)
    
    probs = outputs.logits_per_image.softmax(dim=1)[0]
    top_prob, top_idx = probs.max(0)
    
    if top_prob.item() < confidence_threshold:
        return None
    
    img_width, img_height = image.size
    
    return {
        'class': INGREDIENT_CANDIDATES[top_idx],
        'confidence': top_prob.item(),
        'width': img_width * 0.6,
        'height': img_height * 0.6,
        'x': img_width / 2,
        'y': img_height / 2,
        'detection_method': 'CLIP_single'
    }

print("✓ Single-ingredient detection function defined")

✓ Single-ingredient detection function defined


In [7]:
def detect_multiple_ingredients_clip(image_path: str, 
                                     object_threshold: float = 0.7,
                                     ingredient_threshold: float = 0.15) -> List[Dict]:
    """Detect multiple ingredients using DETR + CLIP"""
    image = Image.open(image_path).convert('RGB')
    
    inputs = detr_processor(images=image, return_tensors="pt")
    outputs = detr_model(**inputs)
    
    target_sizes = torch.tensor([image.size[::-1]])
    results = detr_processor.post_process_object_detection(
        outputs, 
        target_sizes=target_sizes, 
        threshold=object_threshold
    )[0]
    
    detected_ingredients = []
    
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box_coords = [int(i) for i in box.tolist()]
        x1, y1, x2, y2 = box_coords
        
        cropped = image.crop((x1, y1, x2, y2))
        
        clip_inputs = clip_processor(
            text=INGREDIENT_CANDIDATES,
            images=cropped,
            return_tensors="pt",
            padding=True
        )
        
        with torch.no_grad():
            clip_outputs = clip_model(**clip_inputs)
        
        probs = clip_outputs.logits_per_image.softmax(dim=1)[0]
        top_prob, top_idx = probs.max(0)
        
        if top_prob.item() >= ingredient_threshold:
            detected_ingredients.append({
                'class': INGREDIENT_CANDIDATES[top_idx],
                'confidence': top_prob.item(),
                'width': x2 - x1,
                'height': y2 - y1,
                'x': (x1 + x2) / 2,
                'y': (y1 + y2) / 2,
                'bbox': box_coords,
                'detection_method': 'DETR+CLIP_multi'
            })
    
    return detected_ingredients

print("✓ Multi-ingredient detection function defined")

✓ Multi-ingredient detection function defined


In [8]:
def consolidate_detections(detected: List[Dict]) -> Dict:
    """Consolidate multiple detections into ingredient map"""
    ingredient_map = {}
    
    for detection in detected:
        ing_name = detection['class']
        if ing_name not in ingredient_map:
            ingredient_map[ing_name] = {
                'name': ing_name,
                'confidence': detection['confidence'],
                'count': 1,
                'total_area': detection['width'] * detection['height']
            }
        else:
            ingredient_map[ing_name]['confidence'] = max(
                ingredient_map[ing_name]['confidence'],
                detection['confidence']
            )
            ingredient_map[ing_name]['count'] += 1
            ingredient_map[ing_name]['total_area'] += detection['width'] * detection['height']
    
    unique_ingredients = list(ingredient_map.keys())
    combined_ingredient = " and ".join(unique_ingredients)
    
    # Get primary ingredient (largest area)
    primary_ing = max(ingredient_map.items(), key=lambda x: x[1]['total_area'])
    primary_name = primary_ing[0]
    
    return {
        'ingredient_map': ingredient_map,
        'unique_ingredients': unique_ingredients,
        'combined_ingredient': combined_ingredient,
        'primary_ingredient': primary_name,
        'primary_area': primary_ing[1]['total_area'],
        'primary_confidence': primary_ing[1]['confidence']
    }

print("✓ Consolidation function defined")

✓ Consolidation function defined


## Test Detection

Uncomment to test:

In [10]:
# 測試範例 (使用正確的路徑)
# 方法 1: 使用絕對路徑
test_image = DATA_DIR / "test_images" / "test.jpeg"

# 方法 2: 使用相對路徑 (從 FINAL 資料夾)
# test_image = "../../../data/test_images/test.jpeg"

# 執行測試
detected = detect_multiple_ingredients_clip(str(test_image))
result = consolidate_detections(detected)
print(f"Found: {result['combined_ingredient']}")
print(f"Primary: {result['primary_ingredient']} (confidence: {result['primary_confidence']:.1%})")

Found: Chicken breast and Pork kidney
Primary: Chicken breast (confidence: 45.1%)


---

**Module exports:**
- `INGREDIENT_CANDIDATES` (list)
- `detect_ingredient_clip()` (function)
- `detect_multiple_ingredients_clip()` (function)
- `consolidate_detections()` (function)